In [ ]:
import mlflow
import warnings
import logging
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from pathlib import Path
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.ensemble import RUSBoostClassifier
from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings("ignore", category=UserWarning, module="joblib")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Load and Split data

In [ ]:
# 1. Load data
df = pd.read_csv('data/processed/extracted_features_advanced.csv')


X = df.drop(columns=['label'])
y = df['label']

# 4. Encode labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 5. Split the data (80% train, 20% test)
# Stratify is disabled here due to small sample sizes in specific classes
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
feature_names = X_train.columns.tolist()

In [ ]:
# 6. Scaling (Critical for SVM performance)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Helper functions

In [ ]:
def get_metrics(model, X, y):
    """Returns accuracy, macro precision, and macro recall as a dict."""
    y_pred = model.predict(X)
    return {
        "accuracy":  accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_score":  f1_score(y, y_pred, average="macro", zero_division=0),
        }


def save_confusion_matrix(model, X, y, title, path):
    """Saves a confusion matrix plot to disk and returns the file path."""
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return path

def log_run(name, run_id, metrics, params=None, extra=None):
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"  Run ID : {run_id}")
    if params:
        params_str = " | ".join(f"{k}={v}" for k, v in params.items())
        print(f"  Params : {params_str}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<10}: {v}")
    print(f"  Metrics: acc={metrics['accuracy']:.4f} | "
        f"prec={metrics['precision']:.4f} | "
        f"rec={metrics['recall']:.4f} | "
        f"f1={metrics['f1_score']:.4f} |") 
    print(f"{'─'*50}")

# Setting ML flow

In [ ]:
mlflow.set_tracking_uri(uri=(Path.cwd() / 'mlruns').as_uri())

client = MlflowClient()
exp = client.get_experiment_by_name("Glove readings classification")

if exp is None:
    experiment_id = client.create_experiment(
        name="Glove readings classification",
        tags={"topic":"Glove Readings", "version":"v1"}
    )
else:
    experiment_id = exp.experiment_id

print("Experiment ID:", experiment_id)

# Logistic regression

In [ ]:
lr = LogisticRegression(C=1.0, l1_ratio=0, solver="lbfgs", max_iter=200, random_state=42)
lr.fit(X_train, y_train)

with mlflow.start_run(experiment_id=experiment_id, run_name="logistic-regression") as run:

    mlflow.log_param("C",       1.0)
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("solver",  "lbfgs")

    train_metrics = get_metrics(lr, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(lr, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.sklearn.log_model(lr,  "model")

    path = save_confusion_matrix(lr, X_val, y_val,
                                title="Logistic Regression — Validation",
                                path="../figures/cm_logistic.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")
    
    lr_run_id = run.info.run_id
    log_run("Logistic Regression", lr_run_id, metrics, params={"C": 1.0, "solver": "lbfgs"})


# Gaussian Naive Bayes

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

with mlflow.start_run(experiment_id=experiment_id, run_name="gaussian-naive-bayes") as run:

    train_metrics = get_metrics(gnb, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(gnb, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.sklearn.log_model(gnb,  "model")
    
    path = save_confusion_matrix(gnb, X_val, y_val,
                                title="Gaussian Naive Bayes — Validation",
                                path="../figures/cm_gnb.png")
    
    mlflow.log_artifact(path, artifact_path="confusion_matrices") 

    log_run("Gaussian Naive Bayes", run.info.run_id, metrics)

# K-Nearest Neighbors (KNN)

In [ ]:
knn_param_grid = {
    'n_neighbors': [1, 2, 3, 4, 5, 10, 15, 20],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}


knn_base = KNeighborsClassifier()

grid_search = GridSearchCV(knn_base, knn_param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_knn = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))


with mlflow.start_run(experiment_id=experiment_id, run_name="knn-sweep") as run:
    mlflow.log_param("n_neighbors", grid_search.best_params_["n_neighbors"])
    mlflow.log_param("algorithm", grid_search.best_params_["algorithm"])

    train_metrics = get_metrics(best_knn, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_knn, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.sklearn.log_model(best_knn, "model")

    path = save_confusion_matrix(best_knn, X_val, y_val,
                                title="KNN — Validation",
                                path="../figures/cm_knn.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("KNN", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# Suport Vector Machine (SVM)

In [ ]:
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
}

svm_base = SVC(kernel="rbf",    C=1.0, decision_function_shape='ovr', probability=True, random_state=42)

grid_search = GridSearchCV(svm_base, svm_param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_svm = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))


with mlflow.start_run(experiment_id=experiment_id, run_name="svm-sweep") as run:
    mlflow.log_param("C", grid_search.best_params_["C"])
    mlflow.log_param("gamma", grid_search.best_params_["gamma"])

    train_metrics = get_metrics(best_svm, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_svm, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.sklearn.log_model(best_svm, "model")

    path = save_confusion_matrix(best_svm, X_val, y_val,
                                title="SVM — Validation",
                                path="../figures/cm_svm.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("SVM", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# Random Forest Classifier

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [3, 5, None],
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))

In [ ]:
with mlflow.start_run(experiment_id=experiment_id, run_name="random-forest-best") as run:
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("cv_accuracy", grid_search.best_score_)

    train_metrics = get_metrics(best_rf, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_rf, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"]) 
    mlflow.sklearn.log_model(best_rf, "model")  

    for name, importance in zip(feature_names, best_rf.feature_importances_):
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        mlflow.log_metric(f"importance_{clean_name}", importance)
        
    path = save_confusion_matrix(best_rf, X_val, y_val,
                                title="Random Forest — Validation",
                                path=f"../figures/cm_random_forest.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")
    

    log_run("Random Forest", run.info.run_id, metrics,params=grid_search.best_params_,extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# XGBoost

In [ ]:
param_grid = {
    "n_estimators": [50, 80, 100, 200, 300],
    "max_depth":    [3, 4, 5, 8, 10, None],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample":    [0.6, 0.8, 0.9, 1.0]
}

xgb_base = xgb.XGBClassifier(
    random_state=42, 
    eval_metric='logloss' 
)

grid_search = GridSearchCV(xgb_base, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))

with mlflow.start_run(experiment_id=experiment_id, run_name="xgboost") as run:
    mlflow.log_param("n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_param("learning_rate", grid_search.best_params_["learning_rate"])
    mlflow.log_param("subsample", grid_search.best_params_["subsample"])

    train_metrics = get_metrics(best_xgb, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_xgb, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.xgboost.log_model(best_xgb, "model")

    path = save_confusion_matrix(best_xgb, X_val, y_val,
                                title="XGBoost — Validation",
                                path="../figures/cm_xgboost.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("XGBoost", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# AdaBoost

In [ ]:
param_grid_adaboost = {
    "n_estimators": [50, 80, 100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2]
}

adaboost_base = AdaBoostClassifier(random_state=42)

grid_search = GridSearchCV(adaboost_base, param_grid_adaboost, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_adaboost = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))

with mlflow.start_run(experiment_id=experiment_id, run_name="adaboost") as run:

    mlflow.log_param("n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("learning_rate", grid_search.best_params_["learning_rate"])
    
    train_metrics = get_metrics(best_adaboost, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])
    
    metrics = get_metrics(best_adaboost, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.sklearn.log_model(best_adaboost, "model")

    path = save_confusion_matrix(best_adaboost, X_val, y_val,
                                title="AdaBoost — Validation",
                                path="../figures/cm_adaboost.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("AdaBoost", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# Balance Random Forest Classifier

In [ ]:
param_grid_brf = {
    "n_estimators": [50, 80, 100, 200, 300],
    "max_depth":    [3, 4, 5, 8, 10, None],
}

brf_base = BalancedRandomForestClassifier(
    sampling_strategy='auto',
    random_state=42
)

grid_search = GridSearchCV(brf_base, param_grid_brf, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_brf = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))

with mlflow.start_run(experiment_id=experiment_id, run_name="balanced-random-forest") as run:
    
    mlflow.log_param("sampling_strategy", "auto")
    mlflow.log_param("n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("max_depth", grid_search.best_params_["max_depth"])

    train_metrics = get_metrics(best_brf, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_brf, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"]) 
    mlflow.sklearn.log_model(best_brf, "model")

    path = save_confusion_matrix(best_brf, X_val, y_val,
                                title="Balanced Random Forest — Validation",
                                path="../figures/cm_brf.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("Balanced Random Forest", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# RUSBoost Classifier

In [ ]:
rus_base = RUSBoostClassifier(
    random_state=42
)
grid_search = GridSearchCV(rus_base, param_grid_adaboost, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rus = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))

with mlflow.start_run(experiment_id=experiment_id, run_name="rusboost") as run:
    
    mlflow.log_params(grid_search.best_params_)

    train_metrics = get_metrics(best_rus, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(best_rus, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.sklearn.log_model(best_rus, "model")

    path = save_confusion_matrix(best_rus, X_val, y_val,
                                title="RUSBoost — Validation",
                                path="../figures/cm_rusboost.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("RUSBoost", run.info.run_id, metrics, params=grid_search.best_params_, extra={"CV Acc": f"{grid_search.best_score_:.4f}"})

# Balanced Bagging Classifier

In [ ]:
smote_bagging_model = BalancedBaggingClassifier(
    estimator=xgb.XGBClassifier(eval_metric='logloss', random_state=42),
    sampler=SMOTE(random_state=42),
    n_estimators=10,  # Trains 10 XGBoost models on 10 different SMOTE-balanced datasets
    random_state=42
)

smote_bagging_model.fit(X_train, y_train)

with mlflow.start_run(experiment_id=experiment_id, run_name="smote-bagging-xgb") as run:
    
    mlflow.log_param("n_estimators", 10)
    mlflow.log_param("sampler", "SMOTE")

    train_metrics = get_metrics(smote_bagging_model, X_train, y_train)
    mlflow.log_metric("train_accuracy",  train_metrics["accuracy"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall",    train_metrics["recall"])

    metrics = get_metrics(smote_bagging_model, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])
    mlflow.log_metric("val_f1",        metrics["f1_score"])
    mlflow.sklearn.log_model(smote_bagging_model, "model") 

    path = save_confusion_matrix(smote_bagging_model, X_val, y_val,
                                title="SMOTE Bagging — Validation",
                                path="../figures/cm_smote_bagging.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    log_run("SMOTE Bagging", run.info.run_id, metrics, params={"n_estimators": 10, "sampler": "SMOTE"}, extra={"CV Acc": f"{smote_bagging_model.score(X_val, y_val):.4f}"})